**Eurovision Voting Blocs Across Eras (1975–2024)**

# 01 - Data Acquisition

**Objectives**
- Download the three public data sources used in the project
- Convert the EurovisionDataset JSON file into the same table format as the main dataset
- Keep a local copy of every raw file in `data/raw/`, so that later notebooks never depend on the internet

**Output:** `data/raw/votes_spijkervet.csv`, `data/raw/contestants_spijkervet.csv`, `data/raw/votes_josago.csv`, `data/raw/dist_cepii.dta`

> Run this notebook from the `src/` folder. Files that already exist are not downloaded again.

In [1]:
import os

import requests
import pandas as pd

## 1. Data Sources

| Source | What it contains | Years used | File |
|---|---|---|---|
| **Spijkervet & Burgoyne, *Eurovision Song Contest Dataset*** (ISMIR 2023, Zenodo 10088892) | Every vote given by every country, per round | 1975–2023 | `votes_spijkervet.csv`, `contestants_spijkervet.csv` |
| **josago97, *EurovisionDataset*** (GitHub) | Same kind of vote records, published as JSON | 2024 only | `votes_josago.csv` |
| **CEPII GeoDist** (Mayer & Zignago, 2011) | Geographic and cultural indicators for every pair of countries | — | `dist_cepii.dta` |

**What is a "vote record"?** In the Grand Final every participating country (the *voter*) gives points to other countries (the *performers*): 12 to its favourite, then 10, 8, 7, 6, 5, 4, 3, 2 and 1. One row of the vote table is one pair *voter → performer* in one round of one year, with the points given. Since 2016 each country gives two separate sets of points, one from a professional jury and one from the public (televote), so the table has three point columns: `total_points`, `jury_points` and `tele_points`.

The main dataset stops in 2023, so the 2024 contest is taken from EurovisionDataset. `contestants_spijkervet.csv` (songs, artists, running order) is downloaded for reference but is not used in the network pipeline.

### 1.1. Paths and URLs

In [2]:
DATA_DIR = os.path.join("..", "data", "raw")
os.makedirs(DATA_DIR, exist_ok=True)

SPIJKERVET_BASE = "https://github.com/Spijkervet/eurovision-dataset/releases/download/2023"
JOSAGO_JSON_URL = "https://raw.githubusercontent.com/josago97/EurovisionDataset/main/dataset/eurovision.json"
CEPII_URL = "http://www.cepii.fr/distance/dist_cepii.dta"

## 2. Download Helper

A small function that streams a file from a URL to disk. A browser-like `User-Agent` header is sent because some servers refuse requests without one.

In [3]:
def download_file(url: str, dest_path: str):
    print(f"Downloading {url} -> {dest_path}...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    resp = requests.get(url, headers=headers, stream=True, timeout=60)
    resp.raise_for_status()
    with open(dest_path, "wb") as f:
        for chunk in resp.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print(f"Successfully saved {dest_path} ({os.path.getsize(dest_path)} bytes)")

## 3. Spijkervet Eurovision Dataset (1975–2023)

The release already provides the votes as a CSV table, so it is downloaded as is.

### 3.1. Votes

In [4]:
dest_votes_spijkervet = os.path.join(DATA_DIR, "votes_spijkervet.csv")
if not os.path.exists(dest_votes_spijkervet):
    download_file(f"{SPIJKERVET_BASE}/votes.csv", dest_votes_spijkervet)
else:
    print(f"Already exists: {dest_votes_spijkervet}")

Already exists: ../data/raw/votes_spijkervet.csv


### 3.2. Contestants

In [5]:
dest_contestants_spijkervet = os.path.join(DATA_DIR, "contestants_spijkervet.csv")
if not os.path.exists(dest_contestants_spijkervet):
    download_file(f"{SPIJKERVET_BASE}/contestants.csv", dest_contestants_spijkervet)
else:
    print(f"Already exists: {dest_contestants_spijkervet}")

Already exists: ../data/raw/contestants_spijkervet.csv


## 4. EurovisionDataset (2024)

This source is a single JSON file with a nested structure:

```
contest (one per year)
└── rounds (semi-finals, final)
    └── performances (one per competing country)
        └── scores: "total", "public" (televote), "jury"
            └── votes: {voter country: points}
```

To use it together with the main dataset, the nested structure is **flattened**: every *(voter, performer)* pair becomes one row, with the same columns as `votes_spijkervet.csv`.

In [6]:
def download_and_convert_josago(dest_path: str):
    print(f"Fetching josago97 dataset from {JOSAGO_JSON_URL}...")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    resp = requests.get(JOSAGO_JSON_URL, headers=headers, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    rows = []
    for contest in data:
        year = contest.get("year")
        contestants_map = {c["id"]: c.get("country", "") for c in contest.get("contestants", [])}
        for rd in contest.get("rounds", []):
            round_name = rd.get("name")
            for perf in rd.get("performances", []):
                to_cid = perf.get("contestantId")
                to_country = contestants_map.get(to_cid, "")
                scores_by_type = {}
                for sc in perf.get("scores", []):
                    scores_by_type[sc.get("name")] = sc.get("votes", {})

                total_votes = scores_by_type.get("total", {})
                tele_votes = scores_by_type.get("public", {})
                jury_votes = scores_by_type.get("jury", {})

                # one row per voter that appears in any of the three score types
                all_voters = set(total_votes.keys()) | set(tele_votes.keys()) | set(jury_votes.keys())
                for voter in all_voters:
                    rows.append({
                        "year": year,
                        "round": round_name,
                        "from_country_id": voter.lower(),
                        "to_country_id": to_country.lower(),
                        "from_country": voter,
                        "to_country": to_country,
                        "total_points": total_votes.get(voter, 0),
                        "tele_points": tele_votes.get(voter, None),
                        "jury_points": jury_votes.get(voter, None)
                    })
    df = pd.DataFrame(rows)
    df.to_csv(dest_path, index=False)
    print(f"Successfully converted and saved josago97 votes -> {dest_path} ({len(df)} rows)")

In [7]:
dest_votes_josago = os.path.join(DATA_DIR, "votes_josago.csv")
if not os.path.exists(dest_votes_josago):
    download_and_convert_josago(dest_votes_josago)
else:
    print(f"Already exists: {dest_votes_josago}")

Already exists: ../data/raw/votes_josago.csv


## 5. CEPII GeoDist

GeoDist describes every ordered pair of countries *(origin, destination)*. Two of its binary variables are used later as country-pair attributes:

- `contig` = 1 if the two countries share a land border;
- `comlang_off` = 1 if they share an official language.

The file is distributed in Stata format (`.dta`) and is read with `pyreadstat` in notebook 02.

In [8]:
dest_dist_cepii = os.path.join(DATA_DIR, "dist_cepii.dta")
if not os.path.exists(dest_dist_cepii):
    download_file(CEPII_URL, dest_dist_cepii)
else:
    print(f"Already exists: {dest_dist_cepii}")

Already exists: ../data/raw/dist_cepii.dta


## 6. Quick Look at the Raw Files

A first check that the files are readable and have the expected columns.

In [9]:
votes_spijkervet = pd.read_csv(dest_votes_spijkervet)
print(f"Spijkervet votes: {votes_spijkervet.shape}, years {votes_spijkervet['year'].min()}–{votes_spijkervet['year'].max()}")
votes_spijkervet.head()

Spijkervet votes: (51354, 9), years 1957–2023


,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
0,1957,final,at,nl,at,nl,6,NaN,NaN
1,1957,final,at,fr,at,fr,0,NaN,NaN
2,1957,final,at,dk,at,dk,0,NaN,NaN
3,1957,final,at,lu,at,lu,3,NaN,NaN
4,1957,final,at,de,at,de,0,NaN,NaN


In [10]:
votes_josago = pd.read_csv(dest_votes_josago)
print(f"EurovisionDataset votes: {votes_josago.shape}, years {votes_josago['year'].min()}–{votes_josago['year'].max()}")
votes_josago[votes_josago["year"] == 2024].head()

EurovisionDataset votes: (51452, 9), years 1957–2024


,year,round,from_country_id,to_country_id,from_country,to_country,total_points,tele_points,jury_points
49953,2024,final,es,ch,ES,CH,18,6.0,12.0
49954,2024,final,mt,ch,MT,CH,18,6.0,12.0
49955,2024,final,ee,ch,EE,CH,19,7.0,12.0
49956,2024,final,cz,ch,CZ,CH,17,7.0,10.0
49957,2024,final,is,ch,IS,CH,12,6.0,6.0


Rows with `total_points = 0` are kept on purpose: they record that a voter *could* have voted for a performer but did not. Notebook 02 needs them to count in how many years each pair of countries met in the Grand Final.

In [11]:
final_votes = votes_spijkervet[votes_spijkervet["round"] == "final"]
print(f"Share of Grand Final rows with 0 points: {(final_votes['total_points'] == 0).mean():.1%}")

Share of Grand Final rows with 0 points: 54.5%
